# ScummVR 3D Room Generator
Convert DOTT room backgrounds into 3D meshes using TRELLIS (Microsoft)

**Instructions:**
1. Go to Runtime → Change runtime type → GPU (T4)
2. Upload your room PNG files (from nutcracker extraction)
3. Run all cells
4. Download the generated GLB files

In [ ]:
# Install TRELLIS
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
!pip install git+https://github.com/microsoft/TRELLIS.git -q
!pip install pillow trimesh -q

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# Upload room backgrounds
from google.colab import files
import os

os.makedirs('rooms', exist_ok=True)
os.makedirs('output', exist_ok=True)

print("Upload your DOTT room PNG files:")
uploaded = files.upload()
for fname in uploaded:
    with open(f'rooms/{fname}', 'wb') as f:
        f.write(uploaded[fname])
    print(f"  Saved: rooms/{fname}")

In [ ]:
# Load TRELLIS pipeline
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import render_utils, postprocessing_utils
from PIL import Image

pipeline = TrellisImageTo3DPipeline.from_pretrained("microsoft/TRELLIS-image-large")
pipeline.cuda()
print("TRELLIS loaded!")

In [ ]:
# Generate 3D for each room
import glob

room_files = sorted(glob.glob('rooms/*.png'))
print(f"Processing {len(room_files)} rooms...\n")

for i, room_path in enumerate(room_files):
    fname = os.path.basename(room_path)
    print(f"[{i+1}/{len(room_files)}] {fname}...")
    
    image = Image.open(room_path)
    
    # Run TRELLIS
    outputs = pipeline.run(image, seed=42)
    
    # Export as GLB mesh
    glb_path = f"output/{fname.replace('.png', '.glb')}"
    glb = postprocessing_utils.to_glb(
        outputs['gaussian'][0],
        outputs['mesh'][0],
        simplify=0.95,  # reduce mesh complexity
        texture_size=1024
    )
    glb.export(glb_path)
    print(f"  → {glb_path} ({os.path.getsize(glb_path) / 1024:.0f} KB)")
    
    # Also save gaussian splat as PLY
    ply_path = f"output/{fname.replace('.png', '.ply')}"
    outputs['gaussian'][0].save_ply(ply_path)
    print(f"  → {ply_path}")
    
    # Free memory between rooms
    torch.cuda.empty_cache()

print(f"\nDone! {len(room_files)} rooms converted.")

In [ ]:
# Download all generated files
import shutil

shutil.make_archive('scummvr_3d_rooms', 'zip', 'output')
files.download('scummvr_3d_rooms.zip')
print("Download started!")

In [ ]:
# Preview a room (renders a turntable video)
from IPython.display import Video
import imageio

# Pick the first room for preview
room_path = room_files[0]
image = Image.open(room_path)
outputs = pipeline.run(image, seed=42)

# Render turntable
video = render_utils.render_video(outputs['gaussian'][0])['color']
imageio.mimsave('preview.mp4', video, fps=30)
Video('preview.mp4')